In [2]:
import ast, json, re
import pandas as pd
import numpy as np


def norm_code(c: str) -> str:
    c = str(c).strip()
    c = c.replace("ICD-9:", "").replace("ICD9:", "").replace("ICD:", "")
    c = c.replace(".", "").replace(" ", "")
    m = re.search(r"\b(\d{3,5})\b", c)
    return m.group(1) if m else ""

def parse_gold_list(s):
    try:
        vals = ast.literal_eval(str(s))
        return sorted({norm_code(v) for v in vals if norm_code(v)})
    except Exception:
        return []

def parse_pred_codes_conf_from_json(s):
    """
    final_diagnoses_json -> (codes, confidences)
    Accepts list of dicts: [{"label":"75139","confidence":0.92}, ...]
    """
    try:
        arr = json.loads(str(s))
        if isinstance(arr, list):
            codes, confs = [], []
            for d in arr:
                if not isinstance(d, dict):
                    continue
                code = norm_code(d.get("label", ""))
                if code:
                    codes.append(code)
                    
                    try:
                        confs.append(float(d.get("confidence", np.nan)))
                    except Exception:
                        confs.append(np.nan)
            
            confs = [c for c in confs if isinstance(c, (int, float)) and not np.isnan(c)]
            return sorted(set(codes)), confs
    except Exception:
        pass
    return [], []

def parse_pred_from_text(s):
    """Regex from final_explanation free text (no confidences available)."""
    text = str(s)
    codes = set()

    for m in re.finditer(r"(?:ICD-?9|ICD)\s*[:#]?\s*([0-9]{3,5}(?:\.[0-9A-Za-z]+)?)",
                         text, flags=re.IGNORECASE):
        codes.add(norm_code(m.group(1)))

    window_hits = re.findall(
        r"(?:diagnos\w*|possible|likely|code|ICD-?9|ICD)\D{0,20}([0-9]{3,5}(?:\.[0-9A-Za-z]+)?)",
        text, flags=re.IGNORECASE
    )
    for w in window_hits:
        codes.add(norm_code(w))

    if not codes:
        for m in re.finditer(r"\b([0-9]{3,5})(?:\.[0-9A-Za-z]+)?\b", text):
            codes.add(norm_code(m.group(1)))

    codes.discard("")
    return sorted(codes)

def prf(pred, gold):
    pset, gset = set(pred), set(gold)
    if not pset and not gset:
        return (1.0, 1.0, 1.0, True, False)
    if not pset:
        return (0.0, 0.0, 0.0, False, False)
    tp = len(pset & gset)
    prec = tp / len(pset)
    rec  = tp / len(gset) if gset else 1.0
    f1   = 0.0 if (prec+rec)==0 else 2*prec*rec/(prec+rec)
    exact = (pset == gset)
    any_hit = tp > 0
    return (prec, rec, f1, exact, any_hit)


df = pd.read_csv("questions_CE_RF_RSN.csv")  

# gold labels
df["gold_icd9"] = df["diagnoses_icd9"].apply(parse_gold_list)


codes_confs = df.get("final_diagnoses_json", pd.Series([""]*len(df))).apply(parse_pred_codes_conf_from_json)
df["pred_from_json"] = [cc[0] for cc in codes_confs]
df["conf_from_json"] = [cc[1] for cc in codes_confs]


df["pred_from_text"] = df["final_explanation"].apply(parse_pred_from_text)


def choose_pred(row):
    if row["pred_from_json"]:
        return row["pred_from_json"], row["conf_from_json"]
    return row["pred_from_text"], []  

chosen = df.apply(choose_pred, axis=1)
df["pred_icd9"] = [c[0] for c in chosen]
df["pred_conf_list"] = [c[1] for c in chosen]


df["final_dx_count"] = df["pred_icd9"].apply(len)
def avg_conf(xs):
    xs = [x for x in xs if isinstance(x,(int,float)) and not np.isnan(x)]
    return float(np.mean(xs)) if xs else np.nan
df["final_dx_avg_conf"] = df["pred_conf_list"].apply(avg_conf)


metrics = df.apply(lambda r: prf(r["pred_icd9"], r["gold_icd9"]), axis=1)
df["prec"], df["recall"], df["f1"], df["exact_match"], df["any_hit"] = zip(*metrics)


macro = {
    "precision_macro":   float(np.nanmean(df["prec"])),
    "recall_macro":      float(np.nanmean(df["recall"])),
    "f1_macro":          float(np.nanmean(df["f1"])),
    "exact_match_rate":  float(np.nanmean(df["exact_match"])),
    "any_hit_rate":      float(np.nanmean(df["any_hit"])),
    "avg_num_final_dx":  float(np.nanmean(df["final_dx_count"])),
    "avg_confidence":    float(np.nanmean(df["final_dx_avg_conf"]))  # averaged over rows with confidences
}

print("=== Macro metrics ===")
for k, v in macro.items():
    print(f"{k}: {v:.4f}")


df.to_csv("recllama_output_with_metrics.csv", index=False, encoding="utf-8-sig")
print("Wrote recllama_output_with_metrics.csv")

=== Macro metrics ===
precision_macro: 0.0100
recall_macro: 0.0033
f1_macro: 0.0050
exact_match_rate: 0.0000
any_hit_rate: 0.0500
avg_num_final_dx: 4.2500
avg_confidence: 0.9089
Wrote recllama_output_with_metrics.csv
